## **Goal**
**Given the ground truth (Unreal Engine) coordinates and camera rotation vector, calculate:**
* Pointing vector (shoulder joint -> wrist joint) in camera coordinates
* Distance from shoulder joint to camera
* Angles between shoulder-wrist and shoulder-camera vectors

### **Setup**

```
conda create -n fbv-gt-calc python=3.10
conda activate fbv-gt-calc
conda install numpy
conda install ipykernel=6.29.5
```

In [1]:
import numpy as np
import json

### **Given**

In [2]:
# Load JSON Metadata

JSON_A_PATH = "../../../dataset/metadata_A/00003A.json"

with open(JSON_A_PATH, 'r') as file:
    raw_data = json.load(file)

data = {
    'RS': np.array([raw_data['Right Shoulder Coords'][axis] for axis in ['x', 'y', 'z']]),
    'RW': np.array([raw_data['Right Wrist Coords'][axis] for axis in ['x', 'y', 'z']]),
    'CC': np.array([raw_data['Camera Coords'][axis] for axis in ['x', 'y', 'z']]),
    'CR': np.array([raw_data['Camera Rotator'][axis] for axis in ['x', 'y', 'z']]),
}

print(data)

{'RS': array([-2285.95959859,   -37.81550278,   128.70578003]), 'RW': array([-2278.52957027,   -40.82621842,   174.73826095]), 'CC': array([-1830.92814321,   -50.69981304,   682.60686303]), 'CR': array([-0.6346713 ,  0.01797085, -0.77257323])}


In [3]:
# # Keypoint Global Coordinates

# shoulder_coords_global = np.array([-4581.538, -185.471, 141.476])
# wrist_coords_global = np.array([-4528.306, -185.768, 141.553])
# camera_coords_global = np.array([-4079.628, -319.957, 441.476])
# camera_forward_global = np.array([-0.837, 0.224, -0.500])

In [4]:
# # Keypoint Global Coordinates

# shoulder_coords_global = np.array([-4581.538 ,-185.471 ,141.47])
# wrist_coords_global = np.array([-4531.860 ,-183.063 ,122.502])
# camera_coords_global = np.array([-4581.538 ,414.529 ,141.476])
# camera_forward_global = np.array([0.000 ,-1.000 ,0.000])

In [5]:
# Keypoint Global Coordinates

shoulder_coords_global = data['RS']
wrist_coords_global = data['RW']
camera_coords_global = data['CC']
camera_forward_global = data['CR']

In [6]:
# Assumed Constants

camera_coords = np.array([0, 0, 0])
world_up = np.array([0, 0, 1])

In [7]:
# Expected Results

expected_distance = 600.0 # cm
expected_yaw = -15.0      # deg
expected_pitch = 30.0     # deg
expected_angular_separation = np.sqrt(expected_yaw**2 + expected_pitch**2)

### **Calculations**

In [8]:
# # Create Rotation Matrix

# camera_right_global = np.cross(camera_forward_global, world_up)
# camera_right_global /= np.linalg.norm(camera_right_global)

# camera_up_global = np.cross(camera_forward_global, camera_right_global)
# camera_up_global /= np.linalg.norm(camera_up_global)

# rotation_matrix = np.vstack((camera_forward_global,
#                              camera_right_global,
#                              camera_up_global))

# print(rotation_matrix)

In [9]:
# Create Rotation Matrix

camera_right_global = np.cross(world_up, camera_forward_global) 
camera_right_global /= np.linalg.norm(camera_right_global)

camera_up_global = np.cross(camera_forward_global, camera_right_global)
camera_up_global /= np.linalg.norm(camera_up_global)

rotation_matrix = np.vstack((camera_forward_global,
                             camera_right_global,
                             camera_up_global))
print(rotation_matrix)

[[-0.6346713   0.01797085 -0.77257323]
 [-0.02830386 -0.99959937  0.        ]
 [-0.77226371  0.02186681  0.63492567]]


In [10]:
# Translate To Origin

shoulder_coords = shoulder_coords_global - camera_coords_global
wrist_coords = wrist_coords_global - camera_coords_global

print(shoulder_coords)
print(wrist_coords)

[-455.03145538   12.88431026 -553.901083  ]
[-447.60142706    9.87359461 -507.86860209]


In [11]:
# Apply Rotation Transformation

shoulder_coords = rotation_matrix @ shoulder_coords
wrist_coords = rotation_matrix @ wrist_coords

print(shoulder_coords)
print(wrist_coords)

[ 7.16956094e+02 -1.77635684e-15  5.68434189e-14]
[676.62290074   2.79921094  23.42342793]


### **Analysis**

In [12]:
# Shoulder-Camera Distance

distance = np.linalg.norm(shoulder_coords)
print(f"Distance should be close to {expected_distance} cm:")
print(distance, "cm")

Distance should be close to 600.0 cm:
716.9560939073565 cm


In [13]:
# Shoulder-Wrist Shoulder-Camera Vectors

shoulder_wrist = wrist_coords - shoulder_coords
shoulder_wrist /= np.linalg.norm(shoulder_wrist)

shoulder_camera = camera_coords - shoulder_coords
shoulder_camera /= np.linalg.norm(shoulder_camera)

print(shoulder_wrist)
print(shoulder_camera)

[-0.8631971   0.05990775  0.50130013]
[-1.00000000e+00  2.47763685e-18 -7.92843793e-17]


In [14]:
# Angular Separation

angular_separation_rad = np.arccos(np.clip(np.dot(shoulder_wrist, shoulder_camera), -1.0, 1.0))
angular_separation_deg = np.rad2deg(angular_separation_rad)

print(f"Angular separation should be close to {expected_angular_separation} deg:")
print(angular_separation_deg)


Angular separation should be close to 33.54101966249684 deg:
30.32252882044677


### **Angular Separation Components**
Given the angular separation of the shoulder-wrist and shoulder-camera vectors, and camera pitch angle (which could be
retreived from an onboard gyroscope), calculate the vertical component (pitch/elevation) and the horizontal component
(yaw/azimuth) of the angular separation in global coordinates.

In [15]:
# Camera Pitch (Ground Truth / Gyroscope)

camera_pitch_rad = np.asin(camera_forward_global[-1])
camera_pitch_deg = np.rad2deg(camera_pitch_rad)
print(f"Camera pitch should be close to {expected_pitch} deg:")
print(camera_pitch_deg, "deg")

Camera pitch should be close to 30.0 deg:
-50.58552807569504 deg


In [16]:
# # Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)

# unpitch_matrix = np.array([[np.cos(-camera_pitch_rad), 0, np.sin(-camera_pitch_rad)],
#                          [0, 1, 0],
#                          [-np.sin(-camera_pitch_rad), 0, np.cos(-camera_pitch_rad)]])

# print(unpitch_matrix)

In [17]:
# Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)

c, s = np.cos(-camera_pitch_rad), np.sin(-camera_pitch_rad)
unpitch_matrix = np.array([
    [c,  0, s],
    [0,  1, 0],
    [-s, 0, c]
])
print(unpitch_matrix)

[[ 0.63492567  0.          0.77257323]
 [ 0.          1.          0.        ]
 [-0.77257323  0.          0.63492567]]


In [18]:
# Un-Pitch Vectors

shoulder_wrist_gravity = unpitch_matrix @ shoulder_wrist
shoulder_wrist_gravity /= np.linalg.norm(shoulder_wrist_gravity)

shoulder_camera_gravity = unpitch_matrix @ shoulder_camera
shoulder_camera_gravity /= np.linalg.norm(shoulder_camera_gravity)

print(shoulder_wrist_gravity)
print(shoulder_camera_gravity)

[-0.16077494  0.05990775  0.9851713 ]
[-6.34925672e-01  2.47763685e-18  7.72573227e-01]


In [19]:
# Yaw & Pitch Components

shoulder_wrist_gravity_yaw = np.rad2deg(np.atan2(shoulder_wrist_gravity[1], shoulder_wrist_gravity[0]))
shoulder_wrist_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_wrist_gravity[-1], -1.0, 1.0)))

shoulder_camera_gravity_yaw = np.rad2deg(np.atan2(shoulder_camera_gravity[1], shoulder_camera_gravity[0]))
shoulder_camera_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_camera_gravity[-1], -1.0, 1.0)))

delta_yaw = (shoulder_wrist_gravity_yaw - shoulder_camera_gravity_yaw + 180) % 360 - 180
delta_pitch = shoulder_wrist_gravity_pitch - shoulder_camera_gravity_pitch

print(delta_yaw)
print(delta_pitch)

-20.436334210411246
29.535144612285535
